# 🧬 AAontology Z-scored Tensor Generator
Este notebook extrae las escalas fisicoquímicas de **AAontology**, calcula un valor representativo por aminoácido para cada categoría, aplica normalización **z-score** y exporta el resultado como un tensor de PyTorch `(20, 8)` optimizado para pipelines de machine learning o embeddings de proteínas.

## 📦 1. Instalación de dependencias

In [2]:
pip install pandas==2.2.2 --upgrade --force-reinstall

  Using cached pandas-2.2.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (19 kB)
  Using cached numpy-2.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached pytz-2026.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached pandas-2.2.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.7 MB)
Using cached numpy-2.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
Using cached pytz-2026.2-py2.py3-none-any.whl (510 kB)
Using cached tzdata-2026.3-py2.py3-none-any.whl (348 kB)
Using cached six-1.17.0-py2.py3-none-any.whl (11 kB)
  Attempting uninstall: pytz
    Found existing installation: pytz 2026.2
    Uninsta

In [1]:
!pip install -q aaanalysis

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.


## 🔧 2. Imports y configuración inicial

In [18]:
import torch
import pandas as pd
import numpy as np
import aaanalysis as aa
from scipy.stats import zscore

# --- Path ---
OUTPUT_PATH = '/content/drive/MyDrive/Experimentos/ProtFlash/MeanPoolingAAntology/AAntology/aa_ontology_zscored.pt'

AA_ORDER = list('ACDEFGHIKLMNPQRSTVWY')  # Orden canónico de aminoácidos

# Categorías
ACTUAL_CATEGORIES = [
    'ASA/Volume',
    'Composition',
    'Conformation',
    'Energy',
    'Polarity',
    'Shape',
    'Structure-Activity',
    'Others',
]

## 📊 3. Carga de datos oficiales

In [19]:
df_scales = aa.load_scales(name='scales')      # Matriz (20 AA × ~586 escalas)
df_cat    = aa.load_scales(name='scales_cat')  # Metadata de categorías

print(f"✅ df_scales shape: {df_scales.shape}")
print(f"✅ df_cat shape   : {df_cat.shape}")
print(f"✅ Categorías disponibles: {df_cat['category'].unique().tolist()}\n")

✅ df_scales shape: (20, 586)
✅ df_cat shape   : (586, 5)
✅ Categorías disponibles: ['ASA/Volume', 'Composition', 'Conformation', 'Energy', 'Others', 'Polarity', 'Shape', 'Structure-Activity']



## ⚙️ 4. Pipeline de procesamiento y normalización

In [20]:
def compute_aaontology_zscores(df_scales, df_cat, categories, aa_order=AA_ORDER):
    """
    Para cada categoría de AAontology:
      1. Selecciona todas las escalas que pertenecen a esa categoría.
      2. Rellena NaN con la media de cada escala.
      3. Calcula el valor representativo por aminoácido (media de escalas).
    Luego aplica z-score por columna (independiente por categoría).
    Devuelve tensor float32 de shape (20, n_categorías) y lista de categorías válidas.
    """
    col_means  = []
    valid_cats = []

    for cat in categories:
        scale_ids = df_cat.loc[df_cat['category'] == cat, 'scale_id'].tolist()
        if not scale_ids:
            print(f"⚠️  Sin escalas para: {cat}")
            continue

        subset = df_scales.loc[aa_order, scale_ids]
        subset = subset.fillna(subset.mean())      # Rellena NaN por columna
        mean_per_aa = subset.mean(axis=1).values   # Shape: (20,)

        col_means.append(mean_per_aa)
        valid_cats.append(cat)
        print(f"✅ '{cat}': {len(scale_ids)} escalas procesadas")

    if not col_means:
        raise ValueError("No se encontraron escalas para ninguna categoría.")

    matrix  = np.column_stack(col_means)                              # (20, n_cats)
    zscored = np.apply_along_axis(zscore, 0, matrix, nan_policy='omit')
    zscored = np.nan_to_num(zscored, nan=0.0, posinf=0.0, neginf=0.0)

    return torch.tensor(zscored, dtype=torch.float32), valid_cats

## 🚀 5. Ejecución del pipeline

In [21]:
AAONTOLOGY_ZSCORED, USED_CATEGORIES = compute_aaontology_zscores(
    df_scales, df_cat, ACTUAL_CATEGORIES
)

print(f"\n📐 Tensor shape: {AAONTOLOGY_ZSCORED.shape}")

✅ 'ASA/Volume': 64 escalas procesadas
✅ 'Composition': 58 escalas procesadas
✅ 'Conformation': 224 escalas procesadas
✅ 'Energy': 36 escalas procesadas
✅ 'Polarity': 111 escalas procesadas
✅ 'Shape': 45 escalas procesadas
✅ 'Structure-Activity': 31 escalas procesadas
✅ 'Others': 17 escalas procesadas

📐 Tensor shape: torch.Size([20, 8])


## 💾 6. Guardado y verificación de integridad

In [22]:
# 1. Definimos los 20 AA estándar + 1 Token Especial para X (Padding / Unknown)
amino_acids = AA_ORDER.copy()
if 'X' not in amino_acids:
    amino_acids.append('X')

categories = USED_CATEGORIES
num_aa = len(amino_acids)
num_cat = len(categories)

# 2. Inicializamos el tensor en Ceros absolutos (N, 8)
tensor = torch.zeros((num_aa, num_cat), dtype=torch.float32)

# 3. Llenamos el tensor con los z-scores previamente calculados
for i, aa in enumerate(amino_acids):
    if aa == 'X':
        # El Padding/Unknown 'X' permanece en 0.0 absoluto en todas las categorías
        # (Neutro; No altera la suma durante la media ponderada del pooling)
        continue

    # Copiamos la fila correspondiente del tensor z-scored original
    tensor[i, :] = AAONTOLOGY_ZSCORED[i, :]

# 4. Empaquetar y exportar meticulosamente
data_to_save = {
    "tensor": tensor,
    "amino_acids": amino_acids,
    "categories": categories
}

torch.save(data_to_save, OUTPUT_PATH)

print(f"✅ AAontology Z-Scored actualizado con {num_aa} AAs (incluido token 'X') y {num_cat} categorías.")
print(f"💾 Guardado en: {OUTPUT_PATH}")

# Validación visual
idx_x = amino_acids.index('X')
idx_l = amino_acids.index('L')
print(f"\n🔍 Validación interna AAontology:")
print(f"   Tensor 'L' (Leucina): {tensor[idx_l].tolist()}")
print(f"   Tensor 'X' (Padding): {tensor[idx_x].tolist()} (Debe ser nulo)")

✅ AAontology Z-Scored actualizado con 21 AAs (incluido token 'X') y 8 categorías.
💾 Guardado en: /content/drive/MyDrive/Experimentos/ProtFlash/MeanPoolingAAntology/AAntology/aa_ontology_zscored.pt

🔍 Validación interna AAontology:
   Tensor 'L' (Leucina): [0.37219804525375366, 1.5524089336395264, -0.6076842546463013, -0.6159631609916687, 1.3347209692001343, -0.35584932565689087, 0.21614909172058105, 0.12377438694238663]
   Tensor 'X' (Padding): [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] (Debe ser nulo)


## 👁️ 7. Vista previa de resultados

In [23]:
data = torch.load(OUTPUT_PATH, map_location='cpu', weights_only=True)
print("\n🔄 Verificación tras recargar:")
print(f"   Shape        : {data['tensor'].shape}")
print(f"   Aminoácidos  : {data['amino_acids']}")
print(f"   Categorías   : {data['categories']}")
print(f"   dtype        : {data['tensor'].dtype}")
print(f"   min / max    : {data['tensor'].min().item():.3f} / {data['tensor'].max().item():.3f}")

# --- 8. Vista previa de los valores ---
preview_df = pd.DataFrame(
    data['tensor'].numpy(),
    index=data['amino_acids'],
    columns=data['categories'],
).round(3)
print("\n🔍 Preview (valores z-scored por aminoácido):")
print(preview_df)


🔄 Verificación tras recargar:
   Shape        : torch.Size([21, 8])
   Aminoácidos  : ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y', 'X']
   Categorías   : ['ASA/Volume', 'Composition', 'Conformation', 'Energy', 'Polarity', 'Shape', 'Structure-Activity', 'Others']
   dtype        : torch.float32
   min / max    : -2.498 / 2.210

🔍 Preview (valores z-scored por aminoácido):
   ASA/Volume  Composition  Conformation  Energy  Polarity  Shape  \
A      -1.321        1.312        -0.986  -0.876    -0.381 -1.795   
C      -1.005       -1.739        -2.136   1.747     0.807 -0.792   
D      -0.487        0.337         1.789   1.274    -0.832  0.166   
E       0.558        0.761         0.200  -1.270    -0.699  0.657   
F       0.766       -0.760         0.210   0.880     1.463  0.405   
G      -2.400        1.287         0.246  -0.016    -0.892 -2.498   
H       0.207       -1.254         1.124   0.096    -0.393  0.578   
I       0.410    